In [2]:
#Cell 1
import os
import json
import numpy as np
import SimpleITK as sitk
import matplotlib.pyplot as plt
from scipy.ndimage import distance_transform_edt
from skimage.morphology import skeletonize 
from skimage.measure import label
from skimage.filters import frangi

In [3]:
#Cell 2
# 実際のパスに合わせて書き換えてください
DICOM_DIR = "/Users/tkimura/Desktop/Image data/5396641 IPMN/DICOM-A/ScalarVolume_22"

reader = sitk.ImageSeriesReader()
dicom_names = reader.GetGDCMSeriesFileNames(DICOM_DIR)
reader.SetFileNames(dicom_names)
image = reader.Execute()

# Numpy配列に変換 (z, y, x)
data_np = sitk.GetArrayFromImage(image)
# 位置情報の取得 (x, y, z)
spacing = np.array(image.GetSpacing())
origin = np.array(image.GetOrigin())

print(f"Volume Shape: {data_np.shape}")
print(f"Spacing (mm): {spacing}")
print(f"Origin (mm): {origin}")

Volume Shape: (209, 512, 512)
Spacing (mm): [0.625 0.625 1.25 ]
Origin (mm): [-163. -160. -260.]


In [4]:
#Cell 3
# Frangiフィルタの適用
# sigmas: 抽出したい血管の半径(pixel)の範囲を指定します
print("Applying Frangi filter... (This may take a minute)")
vessel_enhanced = frangi(data_np, sigmas=range(1, 4, 1), black_ridges=False)

# 二値化処理 (しきい値は結果を見て調整してください)
# ここでは自動判定の一例として、最大値の10%以上を血管としています
threshold = vessel_enhanced.max() * 0.1
binary_vessel = (vessel_enhanced > threshold).astype(np.uint8)

print(f"Binarization complete. Threshold: {threshold:.6f}")

Applying Frangi filter... (This may take a minute)
Binarization complete. Threshold: 0.045025


In [6]:
#Cell 4
# 1. 各画素から血管の外側までの距離（半径）を計算
dist_map = distance_transform_edt(binary_vessel)

# 2. 3Dスケルトン化（中心線の抽出）
# --- ここを修正 ---
skeleton = skeletonize(binary_vessel) 
# -----------------

print("Skeletonization and Distance Mapping complete.")

Skeletonization and Distance Mapping complete.


In [7]:
#Cell 5
def extract_clean_polylines(skeleton_np):
    # 連結成分ごとにラベル付け
    labeled_skeleton = label(skeleton_np, connectivity=3)
    polylines = []
    
    for i in range(1, labeled_skeleton.max() + 1):
        z, y, x = np.where(labeled_skeleton == i)
        if len(z) < 5: continue  # ノイズ除去（短すぎるセグメント）
        
        # 簡易的な並べ替え（本来はグラフ探索が必要ですが、ここでは成分ごとに抽出）
        pts = np.column_stack((x, y, z)).tolist()
        polylines.append(pts)
    return polylines

all_segments = extract_clean_polylines(skeleton)
print(f"Extracted {len(all_segments)} vessel segments.")

Extracted 620 vessel segments.


In [8]:
# ボリューム中心（mm）の計算
vol_center_idx = np.array(data_np.shape[::-1]) / 2.0
vol_center_mm = origin + vol_center_idx * spacing

output_data = {
    "metadata": {"unit": "mm", "vessel_count": len(all_segments)},
    "polylines": []
}

for idx, seg in enumerate(all_segments):
    blender_nodes = []
    unity_nodes = []
    
    for pt in seg:
        # 画素座標から物理座標(mm)へ
        pt_np = np.array(pt)
        phys_mm = origin + pt_np * spacing
        # 中心を(0,0,0)にするオフセット処理
        centered_mm = phys_mm - vol_center_mm
        
        # 半径(mm)の取得
        radius_vox = dist_map[int(pt[2]), int(pt[1]), int(pt[0])]
        radius_mm = radius_vox * spacing[0]
        
        # Blender用 (右手系: X, Y, Z)
        blender_nodes.append({
            "pos": centered_mm.tolist(),
            "r": float(radius_mm)
        })
        
        # Unity用 (左手系: X, Z, Y) ※UnityはYが上
        unity_nodes.append({
            "pos": [centered_mm[0], centered_mm[2], centered_mm[1]],
            "r": float(radius_mm)
        })
        
    output_data["polylines"].append({
        "id": idx,
        "blender": blender_nodes,
        "unity": unity_nodes
    })

# ファイル保存
output_path = "vessel_centerline_dual.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(output_data, f, indent=2)

print(f"Saved JSON to: {os.path.abspath(output_path)}")

Saved JSON to: /Users/tkimura/Desktop/Programming/Jupyter/日本地図/vessel_centerline_dual.json
